# Collecting Province-Level OCDS Data
Notebook ini mengambil katalog LPSE langsung dari API master OpenTender, memfilter LPSE level provinsi, lalu mengunduh batch OCDS untuk semua provinsi yang tersedia.


## 0. Imports and Global Configuration


In [2]:
import csv
import json
import re
import time
import zipfile
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display


In [3]:
base_location_url = 'https://pro.opentender.net/api/master/lpse/'
base_export_url = 'https://opentender.net/api/tender/export-ocds-batch/'
years = list(range(2015, 2025))
request_timeout = 120
sleep_seconds = 1

json_root = Path('../data/json')
csv_root = Path('../data/csv')
master_csv_path = csv_root / 'master_data_ocds.csv'
summary_csv_path = csv_root / 'raw_data_summary_by_daerah.csv'
all_lspe_catalog_path = csv_root / 'lpse_catalog_all.csv'
province_catalog_path = csv_root / 'lpse_province_catalog.csv'

fieldnames = [
    'lspe_id', 'nama_daerah', 'source_year', 'source_file', 'ocid', 'release_id', 'date',
    'buyer_name', 'buyer_id', 'tender_id', 'tender_title', 'mainProcurementCategory',
    'tender_minValue', 'tender_datePublished', 'tender_status', 'award_id', 'award_title',
    'award_date', 'award_value', 'award_supplier'
]

PROVINCE_NAME_OVERRIDES = {
    'Dki Jakarta': 'DKI Jakarta',
    'Daerah Istimewa Yogyakarta': 'Daerah Istimewa Yogyakarta',
}

json_root.mkdir(parents=True, exist_ok=True)
csv_root.mkdir(parents=True, exist_ok=True)


## 1. Fetch LPSE Catalog and Build Province Targets


In [ ]:
import re
import time

import pandas as pd
import requests

def normalize_whitespace(text: str) -> str:
    return re.sub(r'\s+', ' ', str(text or '')).strip()

# reformatting region so normalized name can be used for grouping and matching with other datasets
def format_region_name(name: str) -> str:
    title_name = normalize_whitespace(name).title()
    return PROVINCE_NAME_OVERRIDES.get(title_name, title_name)

# extract nama provinsi & exceptional handling kalsel
def extract_province_name(lpse_name: str):
    cleaned = normalize_whitespace(lpse_name)
    match = re.match(r'(?i)^lpse\s+provinsi\s+(.+)$', cleaned)
    if match:
        return format_region_name(match.group(1))

    if cleaned.casefold() == 'lpse kalimantan selatan':
        return 'Kalimantan Selatan'

    return None

# fetch LPSE
def fetch_lpse_catalog(api_url: str) -> pd.DataFrame:
    next_url = f'{api_url}?page_size=1000'
    rows = []

    while next_url:
        response = requests.get(next_url, timeout=request_timeout)
        response.raise_for_status()
        payload = response.json()

        page_rows = payload.get('results', []) or []
        rows.extend(page_rows)
        total_count = payload.get('count', len(rows))
        print(f'[catalog] fetched {len(rows):,} / {total_count:,} LPSE rows')

        next_url = (payload.get('links') or {}).get('next')
        time.sleep(0.2)

    catalog = pd.DataFrame(rows)
    if catalog.empty:
        raise RuntimeError('LPSE catalog API returned no rows.')

    catalog = catalog.rename(columns={'code': 'lspe_id', 'name': 'lpse_name_raw'})
    catalog['lspe_id'] = catalog['lspe_id'].astype(str).str.strip()
    catalog['lpse_name_raw'] = catalog['lpse_name_raw'].astype(str).map(normalize_whitespace)
    catalog = catalog.drop_duplicates(subset=['lspe_id']).sort_values(['lpse_name_raw', 'lspe_id']).reset_index(drop=True)
    return catalog


In [ ]:
# call fetch lpse function 
lpse_catalog_all = fetch_lpse_catalog(base_location_url)
lpse_catalog_all.to_csv(all_lspe_catalog_path, index=False)

# take only provinsi level LPSE and extract province name
province_catalog = lpse_catalog_all.copy()
province_catalog['nama_daerah'] = province_catalog['lpse_name_raw'].apply(extract_province_name)
province_catalog = (
    province_catalog.dropna(subset=['nama_daerah'])
    [['lspe_id', 'nama_daerah', 'lpse_name_raw']]
    .sort_values(['nama_daerah', 'lspe_id'])
    .reset_index(drop=True)
)
province_catalog.to_csv(province_catalog_path, index=False)

# error handling not found or else
target_lspe_df = province_catalog[['lspe_id', 'nama_daerah']].copy()

if target_lspe_df.empty:
    raise RuntimeError('Tidak ada LPSE level provinsi yang berhasil dipetakan dari catalog API.')

# result of lpse
print('Target LPSE provinsi:')
display(target_lspe_df)
print(f'Total province LPSE found: {len(target_lspe_df):,}')
print(f'Years covered: {years[0]} - {years[-1]}')
print('Notebook mode: always redownload and rewrite raw master CSV')
print(f'Saved full catalog     : {all_lspe_catalog_path.resolve()}')
print(f'Saved province catalog : {province_catalog_path.resolve()}')


[catalog] fetched 616 / 616 LPSE rows
Target LPSE provinsi:


,lspe_id,nama_daerah
0,106,Aceh
1,33,Bali
2,99,Banten
3,267,Bengkulu
4,127,DKI Jakarta
5,13,Daerah Istimewa Yogyakarta
6,18,Gorontalo
7,70,Jambi
8,14,Jawa Barat
9,42,Jawa Tengah


Total province LPSE found: 34
Years covered: 2015 - 2024
Notebook mode: always redownload and rewrite raw master CSV
Saved full catalog     : C:\Users\VICTUS\coding\collab\3\data\csv\lpse_catalog_all.csv
Saved province catalog : C:\Users\VICTUS\coding\collab\3\data\csv\lpse_province_catalog.csv


## 2. Download Helper


In [ ]:
# formatting name for ocds data
def expected_json_path(lspe_id, year):
    return json_root / lspe_id / f'ocds-data-tender-{year}-{lspe_id}.json'

# download, extract, and save JSON batch with error handling and delay
def download_json_batch(lspe_id, nama_daerah, year):
    # target dir
    target_path = expected_json_path(lspe_id, year)
    target_path.parent.mkdir(parents=True, exist_ok=True)

    # downlaod zip, extract json, save, and cleanup
    zip_path = Path(f'ocds_{lspe_id}_{year}.zip')
    target_url = f'{base_export_url}?year={year}&lpse={lspe_id}'
    print(f'[download] {nama_daerah} ({lspe_id}) {year}')

    response = requests.get(target_url, stream=True, timeout=request_timeout)
    response.raise_for_status()

    with open(zip_path, 'wb') as file_handle:
        for chunk in response.iter_content(chunk_size=8192):
            # write data to file
            if chunk:
                file_handle.write(chunk)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        json_members = [member for member in zip_ref.namelist() if member.endswith('.json')]
        if not json_members:
            raise ValueError(f'No JSON file found inside {zip_path.name}')

        member = json_members[0]
        zip_ref.extract(member, target_path.parent)
        extracted_path = target_path.parent / Path(member).name

        if target_path.exists():
            target_path.unlink()

        extracted_path.rename(target_path)

    zip_path.unlink(missing_ok=True)
    time.sleep(sleep_seconds)
    return target_path


## 3. Download All Province Files


In [7]:
for row in target_lspe_df.itertuples(index=False):
    lspe_id = str(row.lspe_id)
    nama_daerah = row.nama_daerah

    print('=' * 70)
    print(f'DOWNLOADING {nama_daerah} ({lspe_id})')
    print('=' * 70)

    for year in years:
        try:
            download_json_batch(lspe_id, nama_daerah, year)
        except Exception as exc:
            print(f'[error] {nama_daerah} ({lspe_id}) {year}: {exc}')


DOWNLOADING Aceh (106)
[download] Aceh (106) 2015
[download] Aceh (106) 2016
[download] Aceh (106) 2017
[download] Aceh (106) 2018
[download] Aceh (106) 2019
[download] Aceh (106) 2020
[download] Aceh (106) 2021
[download] Aceh (106) 2022
[download] Aceh (106) 2023
[download] Aceh (106) 2024
[error] Aceh (106) 2024: 404 Client Error: Not Found for url: https://opentender.net/api/tender/export-ocds-batch/?year=2024&lpse=106
DOWNLOADING Bali (33)
[download] Bali (33) 2015
[download] Bali (33) 2016
[download] Bali (33) 2017
[download] Bali (33) 2018
[download] Bali (33) 2019
[download] Bali (33) 2020
[download] Bali (33) 2021
[download] Bali (33) 2022
[download] Bali (33) 2023
[download] Bali (33) 2024
[error] Bali (33) 2024: 404 Client Error: Not Found for url: https://opentender.net/api/tender/export-ocds-batch/?year=2024&lpse=33
DOWNLOADING Banten (99)
[download] Banten (99) 2015
[download] Banten (99) 2016
[download] Banten (99) 2017
[download] Banten (99) 2018
[download] Banten (99) 

## 4. Convert JSON into One Raw Master CSV


In [ ]:
total_rows_global = 0
summary_rows = []

with open(master_csv_path, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    # implement everything so far
    for row in target_lspe_df.itertuples(index=False):
        # find everything json to extract
        lspe_id = str(row.lspe_id)
        nama_daerah = row.nama_daerah
        region_dir = json_root / lspe_id
        files = sorted(region_dir.glob(f'ocds-data-tender-*-{lspe_id}.json'))

        if not files:
            print(f'[warning] no files found for {nama_daerah} ({lspe_id})')
            summary_rows.append({
                'lspe_id': lspe_id,
                'nama_daerah': nama_daerah,
                'json_files': 0,
                'rows': 0,
                'min_award_date': None,
                'max_award_date': None,
            })
            continue

        print('=' * 70)
        print(f'PROCESSING {nama_daerah} ({lspe_id})')
        print('=' * 70)

        rows_per_region = 0
        award_dates = []

        # loop through files and extract data
        for file_path in files:
            try:
                payload = json.loads(file_path.read_text(encoding='utf-8'))
            except Exception as exc:
                print(f'[warning] failed reading {file_path.name}: {exc}')
                continue

            source_year = file_path.stem.split('-')[-2]
            for release in payload.get('releases', []):
                tender = release.get('tender', {}) or {}
                buyer = release.get('buyer', {}) or {}
                awards = release.get('awards', []) or []

                for award in awards:
                    suppliers = award.get('suppliers', []) or []
                    supplier_names = ', '.join([supplier.get('name', '') for supplier in suppliers if supplier.get('name')]) or None

                    writer.writerow({
                        'lspe_id': lspe_id,
                        'nama_daerah': nama_daerah,
                        'source_year': source_year,
                        'source_file': file_path.name,
                        'ocid': release.get('ocid'),
                        'release_id': release.get('id'),
                        'date': release.get('date'),
                        'buyer_name': buyer.get('name'),
                        'buyer_id': buyer.get('id'),
                        'tender_id': tender.get('id'),
                        'tender_title': tender.get('title'),
                        'mainProcurementCategory': tender.get('mainProcurementCategory'),
                        'tender_minValue': (tender.get('minValue') or {}).get('amount'),
                        'tender_datePublished': tender.get('datePublished'),
                        'tender_status': tender.get('status'),
                        'award_id': award.get('id'),
                        'award_title': award.get('title'),
                        'award_date': award.get('date'),
                        'award_value': (award.get('value') or {}).get('amount'),
                        'award_supplier': supplier_names,
                    })

                    if award.get('date'):
                        award_dates.append(award.get('date'))

                    rows_per_region += 1
                    total_rows_global += 1

        print(f'Rows written: {rows_per_region:,}')
        print(f'Files found : {len(files)}')

        summary_rows.append({
            'lspe_id': lspe_id,
            'nama_daerah': nama_daerah,
            'json_files': len(files),
            'rows': rows_per_region,
            'min_award_date': min(award_dates) if award_dates else None,
            'max_award_date': max(award_dates) if award_dates else None,
        })


PROCESSING Aceh (106)
Rows written: 12,475
Files found : 9
PROCESSING Bali (33)
Rows written: 1,934
Files found : 9
PROCESSING Banten (99)
Rows written: 3,452
Files found : 9
PROCESSING Bengkulu (267)
Rows written: 1,974
Files found : 9
PROCESSING DKI Jakarta (127)
Rows written: 8,678
Files found : 9
PROCESSING Daerah Istimewa Yogyakarta (13)
Rows written: 4,537
Files found : 10
PROCESSING Gorontalo (18)
Rows written: 1,629
Files found : 10
PROCESSING Jambi (70)
Rows written: 3,001
Files found : 9
PROCESSING Jawa Barat (14)
Rows written: 10,629
Files found : 10
PROCESSING Jawa Tengah (42)
Rows written: 6,038
Files found : 9
PROCESSING Jawa Timur (15)
Rows written: 6,440
Files found : 10
PROCESSING Kalimantan Barat (97)
Rows written: 3,015
Files found : 9
PROCESSING Kalimantan Selatan (181)
Rows written: 2,460
Files found : 9
PROCESSING Kalimantan Tengah (12)
Rows written: 4,120
Files found : 10
PROCESSING Kalimantan Timur (35)
Rows written: 4,053
Files found : 9
PROCESSING Kalimantan U

## 5. Final Summary and Save Diagnostics


In [9]:
summary_df = pd.DataFrame(summary_rows).sort_values(['nama_daerah', 'lspe_id']).reset_index(drop=True)
summary_df.to_csv(summary_csv_path, index=False)

print('RAW COLLECTION SUMMARY')
print('=' * 70)
display(summary_df)

print('Saved files:')
print(f'- Master CSV         : {master_csv_path.resolve()}')
print(f'- Summary CSV        : {summary_csv_path.resolve()}')
print(f'- LPSE Catalog (all) : {all_lspe_catalog_path.resolve()}')
print(f'- LPSE Catalog prov  : {province_catalog_path.resolve()}')
print(f'- Total rows         : {total_rows_global:,}')


RAW COLLECTION SUMMARY


,lspe_id,nama_daerah,json_files,rows,min_award_date,max_award_date
0,106,Aceh,9,12475,2015-03-09T00:00:00.000000Z,2023-10-25T00:00:00.000000Z
1,33,Bali,9,1934,2014-12-10T00:00:00.000000Z,2023-10-10T00:00:00.000000Z
2,99,Banten,9,3452,2015-01-13T00:00:00.000000Z,2023-11-02T00:00:00.000000Z
3,267,Bengkulu,9,1974,2015-02-15T00:00:00.000000Z,2023-11-06T00:00:00.000000Z
4,127,DKI Jakarta,9,8678,2014-12-24T00:00:00.000000Z,2023-10-18T00:00:00.000000Z
5,13,Daerah Istimewa Yogyakarta,10,4537,2015-01-23T00:00:00.000000Z,2024-10-24T14:00:00.000000Z
6,18,Gorontalo,10,1629,2014-12-21T00:00:00.000000Z,2024-09-13T15:30:00.000000Z
7,70,Jambi,9,3001,1-01-01T00:00:00.000000Z,2023-09-30T00:00:00.000000Z
8,14,Jawa Barat,10,10629,2014-12-15T00:00:00.000000Z,2024-11-21T11:00:00.000000Z
9,42,Jawa Tengah,9,6038,2014-11-29T00:00:00.000000Z,2023-10-16T00:00:00.000000Z


Saved files:
- Master CSV         : C:\Users\VICTUS\coding\collab\3\data\csv\master_data_ocds.csv
- Summary CSV        : C:\Users\VICTUS\coding\collab\3\data\csv\raw_data_summary_by_daerah.csv
- LPSE Catalog (all) : C:\Users\VICTUS\coding\collab\3\data\csv\lpse_catalog_all.csv
- LPSE Catalog prov  : C:\Users\VICTUS\coding\collab\3\data\csv\lpse_province_catalog.csv
- Total rows         : 145,922
